In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from ffxt import output
from sympy.polys.subresultants_qq_zz import correct_sign
from torchvision.datasets import CIFAR10

# Datasets & DataLoaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# image => scale (0,1) => normalize (-1,1)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

trainset = CIFAR10(root='./data', train=True, transform=transform, download=True)
testset = CIFAR10(root='./data', train=False, transform=transform, download=True)


In [3]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

In [9]:
# Built the CNN

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # kernel size=2 , stride = 2

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # kernel size=2 , stride = 2

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # kernel size=2 , stride = 2
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256, 120)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # flattening step
        x = self.fc_layers(x)

        return x

In [10]:
model = CNN()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# Training CNN

In [11]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        output = model.forward(images) # Forward propagation
        loss = criterion(output, labels) # loss computation
        loss.backward() # back propagation
        optimizer.step() # update params

        epoch_training_loss += loss.item()

    print(f"epoch {epoch+1} training loss: {epoch_training_loss/len(trainloader)}")

epoch 1 training loss: 1.4777379452877337
epoch 2 training loss: 0.9803427691045015
epoch 3 training loss: 0.7939805157501679
epoch 4 training loss: 0.6699226830926392
epoch 5 training loss: 0.5736809329456075
epoch 6 training loss: 0.48319828397858783
epoch 7 training loss: 0.40361893992594744
epoch 8 training loss: 0.32610639450533313
epoch 9 training loss: 0.2620071285759168
epoch 10 training loss: 0.20637511015605287


In [12]:
# Evaluate our CNN

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        output = model.forward(images)
        _, predicted = torch.max(output, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"Accuracy = {correct_labels/total_labels * 100}%")


Accuracy = 74.96000000000001%
